# Day 19 · 跨程序的握手：認識 A2A Protocol

> 第三部・戰術編排　|　🧠 概念為主，附自製可執行實驗

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 19 - 跨程序的握手：認識 A2A Protocol.md`

## 今天要學會

1. 說得出什麼時候該用、什麼時候不該用 A2A
2. 描述 ADK 的 A2A 工作流程與三個核心能力
3. ⚠️ 分清楚 A2A 跟「開一個 REST API」到底差在哪

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

> 本日需要 `a2a` 套件（`pyproject.toml` 的 `google-adk[a2a]` extra 已包含）。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

import a2a.types as T

print("a2a 套件已就緒，型別數量:", len([n for n in dir(T) if n[0].isupper()]))

google-adk 2.8.0
a2a 套件已就緒，型別數量: 60


## 1. A2A 要解決的問題

到 Day 18 為止，你的 agent 團隊全部活在**同一個 Python 程序**裡。
`sub_agents`、`AgentTool`、`mode` 都是程序內的機制。

一旦另一支 agent 是**別的團隊寫的、跑在別台機器上、甚至不是 Python**，
這些機制全部失效。A2A（Agent-to-Agent Protocol）就是為這件事設計的。

**一句話：A2A 是讓 agent 之間互相呼叫的標準，不是讓你呼叫 agent 的 SDK。**

Day 19 講協定長什麼樣，Day 20 才真的把 server 跑起來。

## 2. 三個核心能力

文章列了三個能力但沒有程式碼。今天用**真的 A2A 型別**（不是自己捏的 dict）
在同一個程序裡把三個能力各演一次。

| # | 能力 | 回答的問題 | 主要型別 |
|---|---|---|---|
| 1 | **Discovery** | 對方是誰？會什麼？怎麼連？ | `AgentCard` |
| 2 | **Task 生命週期** | 這件事做到哪了？ | `Task` / `TaskState` |
| 3 | **串流與非同步回報** | 做的過程能不能邊做邊回報？ | `TaskStatusUpdateEvent` |

這三個能力合起來，才是 A2A 跟「開一個 REST endpoint」的真正差別。

## 3. 能力一：Discovery — Agent Card

Agent Card 是一份**機器可讀的自我介紹**。對方不用讀你的文件就知道你會什麼。

注意它是 protobuf 型別，不是隨便一包 JSON——欄位是被協定定死的。

In [2]:
card = T.AgentCard(
    name="inventory_agent",
    description="查詢商品庫存與補貨狀態",
    version="1.0.0",
    capabilities=T.AgentCapabilities(streaming=True, push_notifications=False),
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[
        T.AgentSkill(
            id="check_stock", name="查庫存",
            description="給一個 SKU，回傳目前庫存數量",
            tags=["inventory"], examples=["A-100 還有幾個？"],
        ),
        T.AgentSkill(
            id="restock_eta", name="查補貨時間",
            description="給一個 SKU，回傳預計補貨日",
            tags=["inventory"], examples=["A-100 什麼時候到貨？"],
        ),
    ],
)

print("=== Agent Card 的完整欄位（由協定定義，不能自己加） ===")
print(" ", [f.name for f in T.AgentCard.DESCRIPTOR.fields])
print()
print("=== 這張卡實際長這樣 ===")
print(card)

=== Agent Card 的完整欄位（由協定定義，不能自己加） ===
  ['name', 'description', 'supported_interfaces', 'provider', 'version', 'documentation_url', 'capabilities', 'security_schemes', 'security_requirements', 'default_input_modes', 'default_output_modes', 'skills', 'signatures', 'icon_url']

=== 這張卡實際長這樣 ===
name: "inventory_agent"
description: "查詢商品庫存與補貨狀態"
version: "1.0.0"
capabilities {
  streaming: true
  push_notifications: false
}
default_input_modes: "text/plain"
default_output_modes: "text/plain"
skills {
  id: "check_stock"
  name: "查庫存"
  description: "給一個 SKU，回傳目前庫存數量"
  tags: "inventory"
  examples: "A-100 還有幾個？"
}
skills {
  id: "restock_eta"
  name: "查補貨時間"
  description: "給一個 SKU，回傳預計補貨日"
  tags: "inventory"
  examples: "A-100 什麼時候到貨？"
}



**`skills` 是給機器看的路由表。** 呼叫方的 agent 讀 `description` 和 `examples`
決定要不要把工作丟過來——跟 Day 17 的 `description` 是路由表是同一件事，
只是這次跨越了程序邊界。

## 4. 能力二：Task 生命週期

這是 A2A 跟 REST 最不一樣的地方。REST 只有「回應」或「錯誤」，
A2A 的一次呼叫會產生一個**有狀態的 Task**，而且狀態不只有成功／失敗。

In [3]:
states = list(T.TaskState.keys())
print("A2A 定義的 Task 狀態共", len(states), "個：\n")
NOTE = {
    "TASK_STATE_SUBMITTED": "已收件，還沒開始",
    "TASK_STATE_WORKING": "進行中",
    "TASK_STATE_COMPLETED": "完成",
    "TASK_STATE_FAILED": "失敗",
    "TASK_STATE_CANCELED": "被取消",
    "TASK_STATE_INPUT_REQUIRED": "⭐ 需要更多輸入——REST 沒有這個概念",
    "TASK_STATE_REJECTED": "拒絕受理",
    "TASK_STATE_AUTH_REQUIRED": "⭐ 需要先通過驗證",
    "TASK_STATE_UNSPECIFIED": "未指定（預設值）",
}
for s in states:
    print(f"  {s:32s} {NOTE.get(s, '')}")

print("\n→ ⭐ 標記的兩個是重點：**遠端 agent 可以回頭跟你要東西**。")
print("   這就是為什麼 A2A 不能用一個 POST 打發掉。")

A2A 定義的 Task 狀態共 9 個：

  TASK_STATE_UNSPECIFIED           未指定（預設值）
  TASK_STATE_SUBMITTED             已收件，還沒開始
  TASK_STATE_WORKING               進行中
  TASK_STATE_COMPLETED             完成
  TASK_STATE_FAILED                失敗
  TASK_STATE_CANCELED              被取消
  TASK_STATE_INPUT_REQUIRED        ⭐ 需要更多輸入——REST 沒有這個概念
  TASK_STATE_REJECTED              拒絕受理
  TASK_STATE_AUTH_REQUIRED         ⭐ 需要先通過驗證

→ ⭐ 標記的兩個是重點：**遠端 agent 可以回頭跟你要東西**。
   這就是為什麼 A2A 不能用一個 POST 打發掉。


## 5. 能力三：串流與非同步回報

長任務不能讓呼叫方乾等。A2A 用 `TaskStatusUpdateEvent` 邊做邊回報進度。

In [4]:
print("TaskStatusUpdateEvent 欄位:", [f.name for f in T.TaskStatusUpdateEvent.DESCRIPTOR.fields])
print("TaskArtifactUpdateEvent 欄位:", [f.name for f in T.TaskArtifactUpdateEvent.DESCRIPTOR.fields])
print()
print("AgentCapabilities 宣告支援哪些:", [f.name for f in T.AgentCapabilities.DESCRIPTOR.fields])
print()
print(f"本卡宣告 streaming={card.capabilities.streaming}, "
      f"push_notifications={card.capabilities.push_notifications}")
print("→ 呼叫方要先看卡片，才知道能不能訂閱進度。")

TaskStatusUpdateEvent 欄位: ['task_id', 'context_id', 'status', 'metadata']
TaskArtifactUpdateEvent 欄位: ['task_id', 'context_id', 'artifact', 'append', 'last_chunk', 'metadata']

AgentCapabilities 宣告支援哪些: ['streaming', 'push_notifications', 'extensions', 'extended_agent_card']

本卡宣告 streaming=True, push_notifications=False
→ 呼叫方要先看卡片，才知道能不能訂閱進度。


## 6. 把三個能力串起來：同程序 mock

現在用一個**假的遠端 agent**把三個能力演一次。
它不開網路埠（那是 Day 20 的事），但用的是**真的 A2A 型別與狀態**。

劇本刻意設計成會走到 `INPUT_REQUIRED`——這正是 REST 做不到的部分。

In [5]:
import asyncio
import uuid

STOCK = {"A-100": 42, "B-200": 0}


class MockRemoteAgent:
    """假裝是別台機器上的 agent。只演協定形狀，不開網路。"""

    def get_agent_card(self) -> T.AgentCard:
        """能力一：Discovery。"""
        return card

    async def send_message(self, text: str, task_id: str | None = None):
        """能力二 + 三：產生 Task，並用串流回報狀態變化。"""
        task_id = task_id or f"task-{uuid.uuid4().hex[:8]}"

        yield T.TaskStatusUpdateEvent(
            task_id=task_id,
            status=T.TaskStatus(state=T.TaskState.TASK_STATE_SUBMITTED),
        )
        await asyncio.sleep(0)
        yield T.TaskStatusUpdateEvent(
            task_id=task_id,
            status=T.TaskStatus(state=T.TaskState.TASK_STATE_WORKING),
        )

        sku = next((k for k in STOCK if k in text), None)
        if sku is None:
            # ⭐ 關鍵：不是失敗，是「我需要你補件」
            yield T.TaskStatusUpdateEvent(
                task_id=task_id,
                status=T.TaskStatus(state=T.TaskState.TASK_STATE_INPUT_REQUIRED),
            )
            return

        yield T.TaskStatusUpdateEvent(
            task_id=task_id,
            status=T.TaskStatus(state=T.TaskState.TASK_STATE_COMPLETED),
        )
        STOCK.setdefault("_last", sku)


remote = MockRemoteAgent()
print("✅ mock 遠端 agent 就緒")

✅ mock 遠端 agent 就緒


In [6]:
async def call_remote(text: str):
    """扮演呼叫方：先讀卡 → 再送訊息 → 依狀態決定下一步。"""
    c = remote.get_agent_card()          # 能力一
    print(f"📇 對方是 {c.name}，會 {[s.name for s in c.skills]}")
    if not c.capabilities.streaming:
        print("   （對方不支援串流，只能等最終結果）")

    last_state = None
    async for ev in remote.send_message(text):        # 能力二 + 三
        name = T.TaskState.Name(ev.status.state)
        print(f"   ▸ {name}")
        last_state = name
    return last_state


print("=== 案例 A：問得很清楚 ===")
await call_remote("A-100 還有幾個？")

print("\n=== 案例 B：沒講是哪個商品 ===")
state = await call_remote("庫存還夠嗎？")
print(f"\n→ 最終狀態是 {state}，不是 FAILED。")
print("   呼叫方該做的是「回頭問使用者要 SKU」，然後用同一個 task_id 續傳。")

=== 案例 A：問得很清楚 ===
📇 對方是 inventory_agent，會 ['查庫存', '查補貨時間']
   ▸ TASK_STATE_SUBMITTED
   ▸ TASK_STATE_WORKING
   ▸ TASK_STATE_COMPLETED

=== 案例 B：沒講是哪個商品 ===
📇 對方是 inventory_agent，會 ['查庫存', '查補貨時間']
   ▸ TASK_STATE_SUBMITTED
   ▸ TASK_STATE_WORKING
   ▸ TASK_STATE_INPUT_REQUIRED

→ 最終狀態是 TASK_STATE_INPUT_REQUIRED，不是 FAILED。
   呼叫方該做的是「回頭問使用者要 SKU」，然後用同一個 task_id 續傳。


**這一段就是 A2A 的價值所在。** 換成 REST，你只會拿到 400 Bad Request，
然後得自己約定一套「什麼錯誤代碼代表要補件」的私有規則——
而那套規則對每個團隊都不一樣。A2A 把它變成協定的一部分。

## 7. 📌 補充：A2A vs 直接開 HTTP API

這是文章裡最有價值、卻只有兩句話的一段。展開成可以照著判斷的表：

| | A2A | 自己開 REST API |
|---|---|---|
| 對方怎麼知道你會什麼 | **Agent Card**，機器可讀 | 讀你的文件（人讀） |
| 「需要補件」怎麼表達 | `INPUT_REQUIRED` 狀態 | 自訂錯誤碼，各家不同 |
| 「要先登入」怎麼表達 | `AUTH_REQUIRED` 狀態 | 401 + 自訂流程 |
| 長任務進度 | `TaskStatusUpdateEvent` 串流 | 自己做 polling 或 webhook |
| 多輪對話 | `context_id` 內建 | 自己管 session |
| 換一家廠商的 agent | 同協定，改 URL | 整個 client 重寫 |
| **成本** | 要多學一套協定 | 你已經會了 |

**判斷準則：**

- 只有你自己團隊在用、而且是**單次問答** → 直接開 REST，別過度設計
- 對方是**別的團隊／別家公司**寫的 agent → A2A（省下無止盡的介面協商）
- 任務會**長時間執行**或**需要多輪往返** → A2A（生命週期是現成的）
- 你想**隨時抽換**背後的 agent 供應商 → A2A

## 8. ⚠️ 什麼時候不該用 A2A

補一段文章沒講、但更容易踩的：

1. **同一個程序裡的 agent 不要用 A2A。** 用 `sub_agents` 或 `AgentTool`（Day 17/18）。
   繞一圈 HTTP 只會多出延遲、序列化成本，還會弄丟 state 共享。
2. **只是想呼叫一個函式，不要用 A2A。** 那是 Function Tool（Day 06）。
3. **對方不是 agent 而是一般 API，不要用 A2A。** 那是 OpenAPI / MCP（Day 07）。
4. **A2A 不會幫你做驗證。** `AUTH_REQUIRED` 只是一個狀態值，
   實際的 token 怎麼發、怎麼驗，還是你自己的事（Day 28）。

In [7]:
# 一張「跨程序連線」的選型速查，把 Day 06 / 07 / 17 / 19 串起來
CHOICES = [
    ("同程序、自己寫的 agent", "sub_agents / AgentTool", "Day 17-18"),
    ("同程序、一個 Python 函式", "FunctionTool", "Day 06"),
    ("外部工具伺服器", "MCP", "Day 07"),
    ("現成的 REST API", "OpenAPI Toolset", "Day 07"),
    ("別人寫的、跨程序的 agent", "A2A", "Day 19-20"),
]
print(f"{'你的情況':<26}{'該用':<26}{'哪一天'}")
print("-" * 66)
for a, b, c in CHOICES:
    print(f"{a:<26}{b:<26}{c}")

你的情況                      該用                        哪一天
------------------------------------------------------------------
同程序、自己寫的 agent            sub_agents / AgentTool    Day 17-18
同程序、一個 Python 函式          FunctionTool              Day 06
外部工具伺服器                   MCP                       Day 07
現成的 REST API              OpenAPI Toolset           Day 07
別人寫的、跨程序的 agent           A2A                       Day 19-20


## 9. ADK 這邊的工作流程

ADK 把 A2A 包成兩個對稱的東西，Day 20 會實際跑：

| 你想做的事 | ADK 給你的 |
|---|---|
| 把自己的 agent **開出去**給別人用 | `to_a2a(agent)` → 一個 Starlette app |
| **消費**別人的 agent | `RemoteA2aAgent(agent_card=...)` |

`RemoteA2aAgent` 拿到之後可以直接當 `sub_agents` 用——
**對主 agent 來說，遠端 agent 跟本地 agent 長得一模一樣。**

In [8]:
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

import inspect

print("to_a2a():")
print("  ", str(inspect.signature(to_a2a))[:150], "...")
sig = inspect.signature(RemoteA2aAgent.__init__)
print("\nRemoteA2aAgent(...) 的必填參數:")
for n, p in sig.parameters.items():
    if n != "self" and p.default is inspect.Parameter.empty and n != "kwargs":
        print(f"  • {n}: {p.annotation}")
print("  ⚠️ agent_card 是**建構參數**，不是 pydantic 欄位——"
      "所以在 model_fields 裡找不到它。")
print("\n→ 注意它也有 mode（Day 18）。")
print("   ⚠️ 但 RemoteA2aAgent 的 task mode 不能當獨立的 workflow 節點，")
print("      只能用在 tool-delegation。這是原始碼裡寫死的限制。")

to_a2a():
   (agent: 'BaseAgent | Workflow', *, host: 'str' = 'localhost', port: 'int' = 8000, protocol: 'str' = 'http', rpc_path: 'str' = '', agent_card: 'AgentCa ...

RemoteA2aAgent(...) 的必填參數:
  • name: str
  • agent_card: Union[AgentCard, str]
  ⚠️ agent_card 是**建構參數**，不是 pydantic 欄位——所以在 model_fields 裡找不到它。

→ 注意它也有 mode（Day 18）。
   ⚠️ 但 RemoteA2aAgent 的 task mode 不能當獨立的 workflow 節點，
      只能用在 tool-delegation。這是原始碼裡寫死的限制。


## 10. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `ModuleNotFoundError: No module named 'a2a'` | 沒裝 extra，要 `google-adk[a2a]` |
| 同程序的 agent 也用 A2A 串 | 過度設計，改用 `sub_agents`（Day 18） |
| 把 `INPUT_REQUIRED` 當成失敗處理 | 它是「請補件」，要用同一個 `task_id` 續傳 |
| Agent Card 的 `skills` 隨便填 | 那是給對方機器讀的路由表，寫得爛就不會被呼叫 |
| 以為 A2A 附帶身分驗證 | `AUTH_REQUIRED` 只是狀態值，驗證要自己做（Day 28） |
| 想把 `RemoteA2aAgent(mode="task")` 當 workflow 節點 | 不支援，只能 tool-delegation |

## 11. 動手練習

1. 幫第 3 節的 `card` 加上第三個 skill，
   確認 `T.AgentCard.DESCRIPTOR` 的欄位清單**不會**因此變動——體會協定是定死的。
2. 改寫 mock，讓「查補貨時間」這個 skill 走 `AUTH_REQUIRED`，
   呼叫方收到之後印出「請先登入」。
3. 把 mock 的 `send_message` 改成中途 `yield` 三次 `WORKING`
   （模擬長任務），觀察呼叫方的輸出變化。
4. 用第 8 節的速查表，替你手上真實的專案挑一種連線方式，寫下理由。

## 本日回顧

- **A2A 是 agent 之間的標準，不是你呼叫 agent 的 SDK。** 同程序就別用它。
- 三個核心能力：**Discovery（Agent Card）／Task 生命週期／串流回報**。
- ⚠️ **`INPUT_REQUIRED` 與 `AUTH_REQUIRED` 是 A2A 勝過 REST 的關鍵**——
  遠端 agent 可以回頭跟你要東西，而這是協定的一部分，不是各家自訂的錯誤碼。
- **Agent Card 的 `skills` 是給機器讀的路由表**，跟 Day 17 的 `description` 同一個道理。
- **判斷準則**：對方是別人寫的、任務會長跑或多輪往返 → A2A；否則多半不需要。
- ⚠️ **A2A 不含身分驗證**，`AUTH_REQUIRED` 只是個狀態值。

---
**下一天 → `../day20_a2a_exposing_consuming/`**